Find the total number of downloads for paying and non-paying users by date. Include only records where non-paying customers have more downloads than paying customers. The output should be sorted by earliest date first and contain 3 columns date, non-paying downloads, paying downloads. 

Note: In Oracle you should use "date" when referring to date column (reserved keyword).

In [0]:
CREATE TABLE ska_catalog.bronze.ms_user_dimension (user_id INT PRIMARY KEY,acc_id INT);
INSERT INTO ska_catalog.bronze.ms_user_dimension (user_id, acc_id) VALUES (1, 101),(2, 102),(3, 103),(4, 104),(5, 105);

CREATE TABLE ska_catalog.bronze.ms_acc_dimension (acc_id INT PRIMARY KEY,paying_customer VARCHAR(10));
INSERT INTO ska_catalog.bronze.ms_acc_dimension (acc_id, paying_customer) VALUES (101, 'Yes'),(102, 'No'),(103, 'Yes'),(104, 'No'),(105, 'No');

CREATE TABLE ska_catalog.bronze.ms_download_facts (date TIMESTAMP,user_id INT,downloads INT);
INSERT INTO ska_catalog.bronze.ms_download_facts (date, user_id, downloads) VALUES ('2024-10-01', 1, 10),('2024-10-01', 2, 15),('2024-10-02', 1, 8),('2024-10-02', 3, 12),('2024-10-02', 4, 20),('2024-10-03', 2, 25),('2024-10-03', 5, 18);

In [0]:
-- Find the total number of downloads for paying and non-paying users by date. Include only records where non-paying customers have more downloads than paying customers. The out   put should be sorted by earliest date first and contain 3 columns date, non-paying downloads, paying downloads.
SELECT
  date_format(c.date, 'yyyy-MM-dd') AS `DATE_FORMAT`, 
  SUM(CASE WHEN b.paying_customer = 'No' THEN c.downloads ELSE 0 END) AS `Non_paying Downloads`,
  SUM(CASE WHEN b.paying_customer = 'Yes' THEN c.downloads ELSE 0 END) As `Paying Downloads`
FROM ska_catalog.bronze.ms_user_dimension a
JOIN ska_catalog.bronze.ms_acc_dimension b
  ON a.acc_id = b.acc_id
JOIN ska_catalog.bronze.ms_download_facts c
  ON a.user_id = c.user_id
GROUP BY DATE_FORMAT
HAVING
  SUM(CASE WHEN b.paying_customer = 'No' THEN c.downloads ELSE 0 END) 
>
  SUM(CASE WHEN b.paying_customer = 'Yes' THEN c.downloads ELSE 0 END)
ORDER BY DATE_FORMAT ASC;